# Automated Test Runner — Single Course

Creates a **single multi-task Lakeflow Job** for one course, with optional QA checks,
so the full run is visible in one timeline.

**Run structure:**
- Course notebooks run in the dependency order you define in `COURSE_TASKS`.
- **QA Content Checker** can run independently (no dependencies) alongside the course tasks.
- The separate **Course-specific inputs** cell makes it easy to reuse this notebook for another course.

```
 QA Check      :  qa_content_checker  (optional, parallel)
 Course Tasks  :  task_01  ──►  task_02  ──►  task_03
```

**What it does**
1. Reads course-specific settings from a dedicated configuration cell (`COURSE_NAME`, `LAB_NOTEBOOKS`, `COURSE_TASKS`).
2. Auto-fills all `<FILL_IN>` placeholders in the configured lab notebooks using the inline solution blocks.
3. Resolves all notebook paths relative to this notebook's location.
4. Creates (or re-creates) a single persistent Lakeflow Job for the configured course.
5. Optionally adds `qa_content_checker` as an **independent task** (no dependencies).
6. Triggers the job with `jobs.run_now` and polls until completion.
7. Appends one result row per task to a generic results table.
8. Displays a combined summary table with clickable `run_page_url` links.
9. Creates and publishes a **Lakeview dashboard** from the run results and QA findings.
10. Raises if any task failed — triggering Lakeflow Job failure email.

## Configuration

In [0]:
%run ./Configuration

## Course-specific inputs

In [0]:
# ── Module 3 Setup fix ────────────────────────────────────────────────────────
# Classroom-Setup-3-demo (cell 2) tries to CREATE CATALOG with a MANAGED LOCATION
# pointing to an S3 external location that doesn't exist in the lab environment.
# Fix: remove the MANAGED LOCATION clause so the catalog uses metastore default storage.
_M03_SETUP = "../Classroom-Setup-3-demo"

CELLS_TO_REPLACE = [
    {
        "relative_path"   : _M03_SETUP,
        "cell_index"      : 2,    # "Create Demo Catalog and Schema" cell
        "original_text"   : "CREATE CATALOG IF NOT EXISTS instructor_interop_demo\n  MANAGED LOCATION 's3://data-interoperability-with-unity-catalog-dev-metastore/unitycatalog/demo/instructor_interop_demo';",
        "replacement_text": "CREATE CATALOG IF NOT EXISTS instructor_interop_demo;",
    },
]

In [0]:
# ── Auto-Fill Lab Notebooks ──────────────────────────────────────────────────
# This script finds all <FILL_IN> placeholders in lab notebooks and replaces
# them with solution code extracted from the nearby solution blocks.
#
# HOW IT WORKS:
# 1. Exports each lab notebook as .ipynb JSON
# 2. Scans code cells for <FILL_IN> markers
# 3. Looks ahead in subsequent markdown cells for <code>...</code> blocks
# 4. Extracts the solution and replaces the original cell source
# 5. Re-imports the patched notebook
#
# Safe to re-run: it only patches cells that still contain <FILL_IN>.
# ──────────────────────────────────────────────────────────────────────────────

import base64
import json
import os
import re
import html as html_lib
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat, Language

w = WorkspaceClient()

# Derive course_root from this notebook's own path
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
course_root = "/".join(_nb_path.split("/")[:-1])

# LAB_NOTEBOOKS is configured in the course-specific inputs cell above.
# Resolve to absolute workspace paths
lab_abs_paths = [os.path.normpath(f"{course_root}/{p}") for p in LAB_NOTEBOOKS]

# Precompiled pattern to strip solution marker HTML comments
_MARKER_PAT = re.compile(r'\s*<!-{2,}.*?(?:ADD SOLUTION CODE|END SOLUTION CODE).*?-{2,}>\s*$')


def extract_solution_from_source(source):
    """Extract code from a Databricks solution cell.

    Handles:
    - Fenced code blocks: ```python\n...\n``` (used in current labs)
    - HTML div blocks: <div class="code-block-dark">...</div>
    - HTML <code> tags: <code>...</code> (legacy fallback)
    - Strips %md / %md-sandbox magic prefixes before parsing.
    """
    # Strip Databricks magic prefix (%md, %md-sandbox, etc.)
    clean = re.sub(r'^%md[a-zA-Z-]*[ \t]*\n', '', source.lstrip(), count=1)

    # 1. Fenced code block (most common in current labs)
    match = re.search(r'```(?:python|sql|scala|r|sh)?\s*\n(.*?)\n[ \t]*```', clean, re.DOTALL)
    if match:
        code = match.group(1)
        lines = code.split('\n')
        lines = [l for l in lines if not _MARKER_PAT.match(l)]
        while lines and not lines[0].strip():
            lines.pop(0)
        while lines and not lines[-1].strip():
            lines.pop()
        return '\n'.join(lines) if lines else None

    # 2. HTML div blocks used by expandable help sections in current labs
    match = re.search(r'<div[^>]*class="[^"]*code-block[^"]*"[^>]*>\s*\n?(.*?)\n?\s*</div>', clean, re.DOTALL)
    if match:
        code = html_lib.unescape(match.group(1))
        code = re.sub(r'<br\s*/?>', '\n', code)
        lines = code.split('\n')
        lines = [l for l in lines if not _MARKER_PAT.match(l)]
        while lines and not lines[0].strip():
            lines.pop(0)
        while lines and not lines[-1].strip():
            lines.pop()
        return '\n'.join(lines) if lines else None

    # 3. HTML <code> tags (fallback for older lab format)
    match = re.search(r'<code[^>]*>\s*\n?(.*?)\n?\s*</code>', clean, re.DOTALL)
    if match:
        code = html_lib.unescape(match.group(1))
        lines = code.split('\n')
        lines = [l for l in lines if not _MARKER_PAT.match(l)]
        while lines and not lines[0].strip():
            lines.pop(0)
        while lines and not lines[-1].strip():
            lines.pop()
        return '\n'.join(lines) if lines else None

    return None


def is_solution_candidate(cell):
    """Return True if a cell is a potential answer/solution block."""
    source = ''.join(cell.get("source", []))
    has_code_block = '```' in source or '<code>' in source or 'code-block' in source
    is_markdown_type = cell.get("cell_type") == "markdown"
    is_md_magic = cell.get("cell_type") == "code" and bool(re.match(r'\s*%md', source))
    return (is_markdown_type or is_md_magic) and has_code_block


def fill_notebook(notebook_path):
    """Fill all <FILL_IN> cells in a single notebook. Returns stats."""
    stats = {"path": notebook_path, "blanks_found": 0, "filled": 0, "no_answer": 0}

    # Export notebook as ipynb
    export_resp = w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb_json = json.loads(base64.b64decode(export_resp.content))
    cells = nb_json.get("cells", [])

    for i, cell in enumerate(cells):
        source = ''.join(cell.get("source", []))

        # Check if this cell has FILL_IN markers or is empty (blank lab cell)
        is_fill_in = '<FILL_IN>' in source or '<FILL IN>' in source or '--FILL IN--' in source or 'FILL_IN' in source
        is_empty_code = cell.get("cell_type") == "code" and not source.strip()
        if not is_fill_in and not is_empty_code:
            continue

        # Skip markdown cells (we only fill code cells)
        if cell.get("cell_type") != "code":
            continue

        stats["blanks_found"] += 1

        # Look ahead up to 5 cells for a solution block.
        solution = None
        for j in range(i + 1, min(i + 6, len(cells))):
            next_cell = cells[j]
            next_source = ''.join(next_cell.get("source", []))
            if is_solution_candidate(next_cell):
                solution = extract_solution_from_source(next_source)
                if solution:
                    break
            # Stop looking once we hit a real (non-magic) code cell
            elif next_cell.get("cell_type") == "code" and not re.match(r'\s*%', next_source):
                break

        if solution:
            if solution.strip().startswith('%sql'):
                solution_lines = solution.strip().split('\n')
                cell["source"] = [solution + '\n']
            else:
                cell["source"] = [solution + '\n']
            stats["filled"] += 1
        else:
            stats["no_answer"] += 1
            print(f"  ⚠️  NO ANSWER FOUND for cell {i} in {notebook_path}")

    # Re-import the patched notebook
    if stats["filled"] > 0:
        patched_content = base64.b64encode(json.dumps(nb_json).encode()).decode()
        w.workspace.import_(
            path=notebook_path,
            content=patched_content,
            format=ImportFormat.JUPYTER,
            overwrite=True,
            language=Language.PYTHON,
        )

    return stats


# ── Run the filler ────────────────────────────────────────────────────────────
print("═" * 70)
print("AUTO-FILL LAB NOTEBOOKS")
print("═" * 70)

all_stats = []
for path in lab_abs_paths:
    print(f"\n▶ Processing: {path.split('/')[-1]}")
    try:
        s = fill_notebook(path)
        all_stats.append(s)
        print(f"   Blanks found: {s['blanks_found']}  |  Filled: {s['filled']}  |  No answer: {s['no_answer']}")
    except Exception as e:
        print(f"   ❌ ERROR: {e}")
        all_stats.append({"path": path, "blanks_found": 0, "filled": 0, "no_answer": 0, "error": str(e)})

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
total_blanks = sum(s["blanks_found"] for s in all_stats)
total_filled = sum(s["filled"] for s in all_stats)
total_no_ans = sum(s["no_answer"] for s in all_stats)
print(f"TOTAL:  {total_blanks} blanks found  |  {total_filled} filled  |  {total_no_ans} blockers")
print("═" * 70)

In [0]:
# Apply targeted text replacements to notebook cells before the job runs.
# Resolves relative_path → notebook_path (absolute) inline, same pattern as CELLS_TO_PATCH.
# Uses CELLS_TO_REPLACE defined in cell 6 above.

import os as _os
_nb_path_tr     = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_course_root_tr = "/".join(_nb_path_tr.split("/")[:-1])

for p in CELLS_TO_REPLACE:
    if "notebook_path" not in p:
        p["notebook_path"] = _os.path.normpath(f"{_course_root_tr}/{p['relative_path']}")

text_restores = patch_text_replacements_for_job(CELLS_TO_REPLACE)

In [0]:
# Catalog / schema / table names
# Derive catalog from the current user's login — each lab user owns a UC catalog
# named after the username portion of their email (dots/hyphens replaced with underscores).
_current_user     = spark.sql("SELECT current_user()").collect()[0][0]
_username         = _current_user.split("@")[0].replace(".", "_").replace("-", "_")

RESULTS_CATALOG   = _username   # e.g. labuser15507750_1782991657
RESULTS_SCHEMA    = "default"   # "default" schema is pre-created in every UC catalog
RESULTS_TABLE     = "notebook_run_results"
QA_FINDINGS_TABLE = "qa_content_findings"

# How long to wait for the entire job run (seconds)
TOTAL_RUN_TIMEOUT_SECONDS = 60 * 60 * 2   # 2 hour ceiling
POLL_INTERVAL_SECONDS     = 15

# Job and dashboard names
JOB_NAME       = f"[Course Validation] {COURSE_NAME}"
DASHBOARD_NAME = f"[Course Validation] {COURSE_NAME} Results Dashboard"

# Build the task list from the reusable course configuration above.
TASKS = []

if RUN_QA_CHECKER:
    TASKS.append({
        "task_key": "qa_content_checker",
        "course": "QA",
        "name": QA_TASK_NAME,
        "relative_path": QA_TASK_RELATIVE_PATH,
        "depends_on": [],
        "compute_type": DEFAULT_COMPUTE_TYPE,
    })

for task in COURSE_TASKS:
    TASKS.append({
        "task_key": task["task_key"],
        "course": COURSE_NAME,
        "name": task["name"],
        "relative_path": task["relative_path"],
        "depends_on": task.get("depends_on", []),
        "compute_type": task.get("compute_type", DEFAULT_COMPUTE_TYPE),
    })

## Imports and workspace client

In [0]:
import json
import os
import time
from datetime import datetime, timezone

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.compute import ClusterSpec, Environment
from databricks.sdk.service.jobs import (
    JobEmailNotifications,
    JobEnvironment,
    NotebookTask,
    RunIf,
    RunLifeCycleState,
    RunResultState,
    Task,
    TaskDependency,
)

from pyspark.sql import Row
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

w = WorkspaceClient()

## Resolve workspace paths

In [0]:
this_notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
course_root = "/".join(this_notebook_path.split("/")[:-1])

for t in TASKS:
    # os.path.normpath resolves '../' so Databricks gets a clean absolute path
    t["notebook_path"] = os.path.normpath(f"{course_root}/{t['relative_path']}")

# Also resolve CELLS_TO_PATCH paths
# for p in CELLS_TO_PATCH:
#     p["notebook_path"] = os.path.normpath(f"{course_root}/{p['relative_path']}")

# print(f"Course root: {course_root}\n")
for t in TASKS:
    print(f"  [{t['task_key']}]  {t['name']}\n      {t['notebook_path']}")

# if CELLS_TO_PATCH:
#     print(f"\nCells to patch ({len(CELLS_TO_PATCH)}):")
#     for p in CELLS_TO_PATCH:
#         print(f"  nuid: {p['cell_index']}  →  {p['notebook_path']}")

## Ensure results table exists

In [0]:
results_fqn = f"{RESULTS_CATALOG}.{RESULTS_SCHEMA}.{RESULTS_TABLE}"

# Ensure catalog and schema exist before writing tables
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RESULTS_CATALOG}.{RESULTS_SCHEMA}")
print(f"Using catalog/schema: {RESULTS_CATALOG}.{RESULTS_SCHEMA}")

results_schema = StructType([
    StructField("run_timestamp",    TimestampType(), nullable=False),
    StructField("job_id",           LongType(),      nullable=True),
    StructField("job_run_id",       LongType(),      nullable=True),
    StructField("course",           StringType(),    nullable=True),
    StructField("task_key",         StringType(),    nullable=False),
    StructField("demo_name",        StringType(),    nullable=False),
    StructField("notebook_path",    StringType(),    nullable=False),
    StructField("status",           StringType(),    nullable=False),  # PASS / FAIL / TIMEOUT
    StructField("result_state",     StringType(),    nullable=True),
    StructField("life_cycle_state", StringType(),    nullable=True),
    StructField("duration_seconds", DoubleType(),    nullable=True),
    StructField("run_id",           LongType(),      nullable=True),
    StructField("run_page_url",     StringType(),    nullable=True),
    StructField("error_message",    StringType(),    nullable=True),
])

if not spark.catalog.tableExists(results_fqn):
    spark.createDataFrame([], results_schema).write.format("delta").saveAsTable(results_fqn)
    print(f"Created results table: {results_fqn}")
else:
    print(f"Results table exists: {results_fqn}")

In [0]:
# ── Optional early exit ──────────────────────────────────────────────────────
# Set RUN_SETUP_ONLY = True to run only the setup/config half (cells 1–12).
# Leave it False (or just run all cells) to continue with job creation & execution.
# ─────────────────────────────────────────────────────────────────────────────
RUN_SETUP_ONLY = False

if RUN_SETUP_ONLY:
    print("Setup complete. Stopping here — set RUN_SETUP_ONLY = False to continue with job creation.")
    dbutils.notebook.exit("Setup-only run complete.")

## Create (or re-create) the course Lakeflow Job

In [0]:
# Delete any existing job with the same name so we always get a clean definition.

existing = [j for j in w.jobs.list(name=JOB_NAME)]
for j in existing:
    w.jobs.delete(job_id=j.job_id)
    print(f"Deleted existing job: {j.job_id} ({j.settings.name})")

# Build Task objects — compute resource determined per task via compute_type.
job_tasks = []
_warehouse_id_cache: dict = {}  # avoid repeated warehouse API calls for same warehouse
for t in TASKS:
    compute_key  = t.get("compute_type", DEFAULT_COMPUTE_TYPE)
    compute_cfg  = COMPUTE_TYPES[compute_key]
    compute_kind = compute_cfg["type"]

    # Resolve retry settings: per-task override → compute-type default → global default
    _max_retries = t.get("max_retries", TASK_MAX_RETRIES)
    _default_interval = WAREHOUSE_RETRY_INTERVAL_SECONDS if compute_kind == "warehouse" else TASK_RETRY_INTERVAL_SECONDS
    _retry_interval_ms = t.get("retry_interval_seconds", _default_interval) * 1000

    task_kwargs = dict(
        task_key=t["task_key"],
        description=t["name"],
        notebook_task=NotebookTask(notebook_path=t["notebook_path"]),
        depends_on=[TaskDependency(task_key=dep) for dep in t["depends_on"]],
        run_if=RunIf[t["run_if"]] if t.get("run_if") else None,
        max_retries=_max_retries if _max_retries > 0 else None,
        min_retry_interval_millis=_retry_interval_ms if _max_retries > 0 else None,
    )

    if compute_kind == "serverless":
        # compute_type key doubles as environment_key; tasks sharing the same
        # compute_type will reference the same JobEnvironment spec below.
        task_kwargs["environment_key"] = compute_key
    elif compute_kind == "classic_cluster":
        task_kwargs["existing_cluster_id"] = compute_cfg["cluster_id"]
    elif compute_kind == "new_cluster":
        task_kwargs["new_cluster"] = ClusterSpec(**compute_cfg["cluster_config"])
    elif compute_kind == "warehouse":
        from databricks.sdk.service.sql import CreateWarehouseRequestWarehouseType as _WarehouseType
        wh_name = compute_cfg["warehouse_name"]
        if wh_name not in _warehouse_id_cache:
            # Find existing warehouse by name, or create one
            wh_id = next((wh.id for wh in w.warehouses.list() if wh.name == wh_name), None)
            if wh_id is None:
                wh_type = _WarehouseType[compute_cfg.get("warehouse_type", "PRO").upper()]
                new_wh = w.warehouses.create(
                    name=wh_name,
                    cluster_size=compute_cfg.get("cluster_size", "2X-Small"),
                    warehouse_type=wh_type,
                    auto_stop_mins=int(compute_cfg.get("auto_stop_mins", 30)),
                )
                wh_id = new_wh.id
                print(f"  Created warehouse '{wh_name}': {wh_id}")
            else:
                print(f"  Found existing warehouse '{wh_name}': {wh_id}")
            _warehouse_id_cache[wh_name] = wh_id
        # Override notebook_task with warehouse_id — no cluster spec on Task
        task_kwargs["notebook_task"] = NotebookTask(
            notebook_path=t["notebook_path"],
            warehouse_id=_warehouse_id_cache[wh_name],
        )
    else:
        raise ValueError(f"Unknown compute type '{compute_kind}' for task '{t['task_key']}'")

    job_tasks.append(Task(**task_kwargs))

# Build one JobEnvironment per unique serverless compute_type actually used by tasks.
used_compute_keys = {t.get("compute_type", DEFAULT_COMPUTE_TYPE) for t in TASKS}
job_environments = [
    JobEnvironment(
        environment_key=ck,
        spec=Environment(environment_version=COMPUTE_TYPES[ck]["environment_version"]),
    )
    for ck in sorted(used_compute_keys)
    if COMPUTE_TYPES[ck]["type"] == "serverless"
]

created_job = w.jobs.create(
    name=JOB_NAME,
    tasks=job_tasks,
    environments=job_environments or None,
    email_notifications=JobEmailNotifications(
        on_failure=TESTER_EMAILS,
        on_success=TESTER_EMAILS,
    ),
)
job_id = created_job.job_id
print(f"Created Lakeflow Job: {job_id}  ({JOB_NAME})")
print(f"\nTask DAG:")
for t in TASKS:
    ct   = t.get("compute_type", DEFAULT_COMPUTE_TYPE)
    deps = " → depends on: " + ", ".join(t["depends_on"]) if t["depends_on"] else " (starts immediately)"
    print(f"  {t['task_key']}  [compute: {ct}]{deps}")

## Trigger the job run and poll until all tasks complete

In [0]:
run_response  = w.jobs.run_now(job_id=job_id)
job_run_id    = run_response.run_id
run_timestamp = datetime.now(timezone.utc)

print(f"Triggered job run: {job_run_id}")
print(f"Polling every {POLL_INTERVAL_SECONDS}s (timeout={TOTAL_RUN_TIMEOUT_SECONDS}s)...\n")

deadline        = time.time() + TOTAL_RUN_TIMEOUT_SECONDS
final_run       = None
terminal_states = {RunLifeCycleState.TERMINATED, RunLifeCycleState.SKIPPED, RunLifeCycleState.INTERNAL_ERROR}

while time.time() < deadline:
    run = w.jobs.get_run(run_id=job_run_id)
    lc  = run.state.life_cycle_state if run.state else None

    task_states = {
        tk.task_key: (
            tk.state.life_cycle_state.value if tk.state and tk.state.life_cycle_state else "PENDING"
        )
        for tk in (run.tasks or [])
    }
    print(f"  [{datetime.now(timezone.utc).strftime('%H:%M:%S')}] run={lc}  tasks={task_states}")

    if lc in terminal_states:
        final_run = run
        break

    time.sleep(POLL_INTERVAL_SECONDS)
else:
    print("TIMEOUT — cancelling the run.")
    try:
        w.jobs.cancel_run(run_id=job_run_id)
    except Exception:
        pass
    final_run = w.jobs.get_run(run_id=job_run_id)

print(f"\nFinal run state: {final_run.state.life_cycle_state if final_run.state else 'UNKNOWN'}")

In [0]:
# Restore original cell content now that the job has completed.
# CELLS_TO_REPLACE is active (2.2 Demo dependency fix), so restore the text
# replacements. CELLS_TO_PATCH (BEGIN...END wrapping) remains disabled, so its
# restore stays commented out.
restore_text_replacements_after_job(text_restores)
# restore_notebooks_after_job(restores)

## Collect per-task results

In [0]:
task_cfg = {t["task_key"]: t for t in TASKS}

# Deduplicate task runs: keep only the latest attempt per task_key
# (retried tasks appear multiple times with ascending attempt_number).
all_task_runs = final_run.tasks or []
latest_by_key: dict = {}
for tr in all_task_runs:
    attempt = tr.attempt_number if hasattr(tr, 'attempt_number') and tr.attempt_number is not None else 0
    prev = latest_by_key.get(tr.task_key)
    if prev is None or attempt > prev[1]:
        latest_by_key[tr.task_key] = (tr, attempt)
deduped_task_runs = [v[0] for v in latest_by_key.values()]

results = []
for task_run in deduped_task_runs:
    cfg   = task_cfg.get(task_run.task_key, {})
    state = task_run.state

    result_state = state.result_state.value     if state and state.result_state     else None
    life_cycle   = state.life_cycle_state.value  if state and state.life_cycle_state  else None
    error_msg    = state.state_message           if state                             else None

    duration = None
    if task_run.start_time and task_run.end_time:
        duration = float((task_run.end_time - task_run.start_time) / 1000.0)

    if life_cycle == RunLifeCycleState.TERMINATED.value and result_state == RunResultState.SUCCESS.value:
        status = "PASS"
    elif life_cycle in (RunLifeCycleState.SKIPPED.value, RunLifeCycleState.INTERNAL_ERROR.value):
        status = "FAIL"
    elif life_cycle == RunLifeCycleState.TERMINATED.value:
        status = "FAIL"
    else:
        status = "TIMEOUT"

    results.append({
        "job_id":           job_id,
        "job_run_id":       job_run_id,
        "course":           cfg.get("course", ""),
        "task_key":         task_run.task_key,
        "demo_name":        cfg.get("name", task_run.task_key),
        "notebook_path":    cfg.get("notebook_path", ""),
        "status":           status,
        "result_state":     result_state,
        "life_cycle_state": life_cycle,
        "duration_seconds": duration,
        "run_id":           task_run.run_id,
        "run_page_url":     task_run.run_page_url,
        "error_message":    error_msg,
    })

# Sort back into configured TASKS order
order = {t["task_key"]: i for i, t in enumerate(TASKS)}
results.sort(key=lambda r: order.get(r["task_key"], 999))

## Append results to Delta + display summary

In [0]:
rows       = [Row(run_timestamp=run_timestamp, **r) for r in results]
results_df = spark.createDataFrame(rows, schema=results_schema)

results_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(results_fqn)

print(f"Appended {results_df.count()} rows to {results_fqn}\n")

# Summary for the configured single-course run
summary_groups = [COURSE_NAME]
if RUN_QA_CHECKER:
    summary_groups.append("QA")

for group_name in summary_groups:
    group_results = [r for r in results if r["course"] == group_name]
    if not group_results:
        continue

    passed = sum(1 for r in group_results if r["status"] == "PASS")
    failed = sum(1 for r in group_results if r["status"] != "PASS")
    icon = "✅" if failed == 0 else "❌"
    label = "QA Checks" if group_name == "QA" else COURSE_NAME
    print(f"  {icon} {label}:  {passed}/{len(group_results)} passed")

print(f"\nOverall: {sum(1 for r in results if r['status'] == 'PASS')}/{len(results)} tasks passed")

display(results_df)

## Create Lakeview Dashboard

In [0]:
import json
from databricks.sdk.service.dashboards import Dashboard

# ── Dashboard Configuration ────────────────────────────────────────────────────
DASHBOARD_FOLDER = "/".join(this_notebook_path.split("/")[:-2])  # Up from CourseRunner/
qa_findings_fqn = f"{RESULTS_CATALOG}.{RESULTS_SCHEMA}.{QA_FINDINGS_TABLE}"
course_label_sql = COURSE_NAME.replace("'", "''")

# ── Dataset SQL (always filters to the latest run) ────────────────────────────
run_filter = f"job_run_id = (SELECT MAX(job_run_id) FROM {results_fqn})"
qa_filter  = f"run_timestamp = (SELECT MAX(run_timestamp) FROM {qa_findings_fqn})"

status_summary_sql = (
    f"SELECT status, COUNT(*) AS task_count "
    f"FROM {results_fqn} WHERE {run_filter} "
    f"GROUP BY status ORDER BY status"
)

task_detail_sql = (
    f"SELECT CASE WHEN course = 'QA' THEN 'QA Checks' ELSE '{course_label_sql}' END AS run_group, "
    f"task_key, demo_name, status, ROUND(duration_seconds, 1) AS duration_seconds, "
    f"run_page_url, error_message "
    f"FROM {results_fqn} WHERE {run_filter} "
    f"ORDER BY CASE WHEN course = 'QA' THEN 0 ELSE 1 END, task_key"
)

qa_sev_sql = (
    f"SELECT severity, COUNT(*) AS issue_count "
    f"FROM {qa_findings_fqn} WHERE {qa_filter} "
    f"GROUP BY severity ORDER BY issue_count DESC"
) if spark.catalog.tableExists(qa_findings_fqn) else (
    "SELECT 'No Data' AS severity, 0 AS issue_count WHERE 1=0"
)

qa_detail_sql = (
    f"SELECT notebook_name, cell_index, issue_type, severity, "
    f"offending_text, suggested_fix, check_source "
    f"FROM {qa_findings_fqn} WHERE {qa_filter} "
    f"ORDER BY severity, notebook_name, cell_index"
) if spark.catalog.tableExists(qa_findings_fqn) else (
    "SELECT '' AS notebook_name, 0 AS cell_index, '' AS issue_type, "
    "'' AS severity, '' AS offending_text, '' AS suggested_fix, '' AS check_source WHERE 1=0"
)

datasets = [
    {
        "name": "ds_status_summary",
        "displayName": "Task Status Summary",
        "query": status_summary_sql,
    },
    {
        "name": "ds_task_detail",
        "displayName": "Task Results Detail",
        "query": task_detail_sql,
    },
]

pages = [
    {
        "name": "pg_runs",
        "displayName": f"{COURSE_NAME} Run Results",
        "layout": [
            {
                "widget": {
                    "name": "w_bar",
                    "title": "Task Status — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_status_summary",
                            "fields": [
                                {"name": "status", "expression": "`status`"},
                                {"name": "task_count", "expression": "`task_count`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "bar",
                        "encodings": {
                            "x": {"fieldName": "status", "displayName": "Status"},
                            "y": {"fieldName": "task_count", "displayName": "Tasks"},
                            "color": {"fieldName": "status", "displayName": "Status"}
                        }
                    }
                },
                "position": {"x": 0, "y": 0, "width": 6, "height": 6}
            },
            {
                "widget": {
                    "name": "w_task_table",
                    "title": "Task Results — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_task_detail",
                            "fields": [
                                {"name": "run_group", "expression": "`run_group`"},
                                {"name": "task_key", "expression": "`task_key`"},
                                {"name": "demo_name", "expression": "`demo_name`"},
                                {"name": "status", "expression": "`status`"},
                                {"name": "duration_seconds", "expression": "`duration_seconds`"},
                                {"name": "run_page_url", "expression": "`run_page_url`"},
                                {"name": "error_message", "expression": "`error_message`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "table",
                        "encodings": {
                            "columns": [
                                {"fieldName": "run_group", "visible": True, "title": "Run Group"},
                                {"fieldName": "task_key", "visible": True, "title": "Task Key"},
                                {"fieldName": "demo_name", "visible": True, "title": "Task Name"},
                                {"fieldName": "status", "visible": True, "title": "Status"},
                                {"fieldName": "duration_seconds", "visible": True, "title": "Duration (s)"},
                                {"fieldName": "run_page_url", "visible": True, "title": "Run URL"},
                                {"fieldName": "error_message", "visible": True, "title": "Error"}
                            ]
                        }
                    }
                },
                "position": {"x": 0, "y": 6, "width": 12, "height": 8}
            }
        ]
    }
]

if RUN_QA_CHECKER:
    datasets.extend([
        {
            "name": "ds_qa_severity",
            "displayName": "QA Issues by Severity",
            "query": qa_sev_sql,
        },
        {
            "name": "ds_qa_detail",
            "displayName": "QA Findings Detail",
            "query": qa_detail_sql,
        },
    ])

    pages.append({
        "name": "pg_qa",
        "displayName": "QA Findings",
        "layout": [
            {
                "widget": {
                    "name": "w_qa_bar",
                    "title": "QA Issues by Severity — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_qa_severity",
                            "fields": [
                                {"name": "severity", "expression": "`severity`"},
                                {"name": "issue_count", "expression": "`issue_count`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "bar",
                        "encodings": {
                            "x": {"fieldName": "severity", "displayName": "Severity"},
                            "y": {"fieldName": "issue_count", "displayName": "Issue Count"}
                        }
                    }
                },
                "position": {"x": 0, "y": 0, "width": 6, "height": 6}
            },
            {
                "widget": {
                    "name": "w_qa_table",
                    "title": "QA Findings Detail — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_qa_detail",
                            "fields": [
                                {"name": "notebook_name", "expression": "`notebook_name`"},
                                {"name": "cell_index", "expression": "`cell_index`"},
                                {"name": "issue_type", "expression": "`issue_type`"},
                                {"name": "severity", "expression": "`severity`"},
                                {"name": "offending_text", "expression": "`offending_text`"},
                                {"name": "suggested_fix", "expression": "`suggested_fix`"},
                                {"name": "check_source", "expression": "`check_source`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "table",
                        "encodings": {
                            "columns": [
                                {"fieldName": "notebook_name", "visible": True, "title": "Notebook"},
                                {"fieldName": "cell_index", "visible": True, "title": "Cell #"},
                                {"fieldName": "issue_type", "visible": True, "title": "Issue Type"},
                                {"fieldName": "severity", "visible": True, "title": "Severity"},
                                {"fieldName": "offending_text", "visible": True, "title": "Offending Text"},
                                {"fieldName": "suggested_fix", "visible": True, "title": "Suggested Fix"},
                                {"fieldName": "check_source", "visible": True, "title": "Source"}
                            ]
                        }
                    }
                },
                "position": {"x": 0, "y": 6, "width": 12, "height": 8}
            }
        ]
    })

spec = {
    "datasets": datasets,
    "pages": pages,
}

# ── Create / re-create the Lakeview Dashboard ─────────────────────────────────
try:
    for d in w.lakeview.list():
        if d.display_name == DASHBOARD_NAME:
            w.lakeview.trash(dashboard_id=d.dashboard_id)
            print(f"Replaced existing dashboard: {d.dashboard_id}")
            break
except Exception:
    pass  # No existing dashboard — proceed to create

dashboard = w.lakeview.create(Dashboard(
    display_name=DASHBOARD_NAME,
    serialized_dashboard=json.dumps(spec),
    parent_path=DASHBOARD_FOLDER,
))
w.lakeview.publish(dashboard_id=dashboard.dashboard_id)

workspace_host = spark.conf.get("spark.databricks.workspaceUrl")
dashboard_url  = f"https://{workspace_host}/dashboardsv3/{dashboard.dashboard_id}"

print(f"✅  Lakeview Dashboard created & published!")
print(f"    Name : {DASHBOARD_NAME}")
print(f"    ID   : {dashboard.dashboard_id}")
print(f"    URL  : {dashboard_url}")

## Fail the notebook if any task failed

In [0]:
failed = [r for r in results if r["status"] != "PASS"]
if failed:
    summary = "\n".join(
        f"  - [{r['task_key']}] {r['demo_name']}: {r['status']} ({r['result_state']}) — {r['run_page_url']}"
        for r in failed
    )
    raise RuntimeError(
        f"{len(failed)} of {len(results)} task(s) failed:\n{summary}"
    )

print(f"All {len(results)} tasks passed.")
dbutils.notebook.exit(json.dumps({
    "total":  len(results),
    "passed": len(results),
    "failed": 0,
}))